# Phase 3 + 5 — `TopologicalEGNN`: Architecture, Training, Ablation

Builds the E(n)-equivariant GNN, trains it on the Phase 4 cluster splits, and
runs the three-arm ablation that decides whether topology earns its place.

**Input:** `phase4_splits.pt` **Output:** trained weights + metrics

---

### Why EGNN rather than a standard GNN

A binding site is defined by 3D geometry, so the model must respect the
symmetries of 3D space: rotating a protein must not change which residues are
predicted as sites. EGNN (Satorras et al., 2021) achieves this by *never* feeding
raw coordinates into the MLPs. Messages depend on coordinates only through
$\|x_i - x_j\|^2$ — a rotation- and translation-invariant scalar — while
coordinates are updated along the equivariant directions $(x_i - x_j)$. The
result: node features are **invariant** and coordinates are **equivariant**,
exactly and by construction, with no data augmentation and no approximation.

This is why Phase 1's centring bug mattered so much. Equivariance is a chain of
guarantees, and the graph is its first link.

### The topology fusion

Concatenating topological features into `h` at layer 0 lets them be diluted by
successive message passing. Instead a **gated cross-attention** block lets each
node *query* its own topological descriptor at every layer:

$$\alpha_i = \sigma\!\big(W_g [h_i \,\|\, t_i]\big), \qquad
  h_i \leftarrow h_i + \alpha_i \odot W_t t_i$$

The gate $\alpha_i$ is learned per-channel, so the network can route around
topology where it is uninformative — and, importantly, **the gate values are
readable after training**, which turns "does topology help?" from a black box
into an inspectable quantity. Since $t_i$ is a rotation-invariant scalar vector,
fusing it this way cannot break equivariance.

## Cell 1 — Setup

In [ ]:
!pip install -q torch-geometric==2.6.1

In [ ]:
import copy, json, math, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

SEED = 1337
torch.manual_seed(SEED); np.random.seed(SEED)


def locate(fname):
    cands = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if root.exists():
            cands.extend(root.rglob(fname))
    if not cands:
        raise FileNotFoundError(f"{fname} not found -- attach the producing notebook output")
    return max(cands, key=lambda p: p.stat().st_size)


blob = torch.load(locate("phase4_splits.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
TOPO_NAMES = blob["topo_names"]
TOPO_DIM = len(TOPO_NAMES)
N_SEQ = int(dataset[0].x.size(1)) - TOPO_DIM      # Phase 1 features

# --- real GPU probe (is_available() lies on unsupported architectures) -------
GPU_OK = False
if torch.cuda.is_available():
    try:
        _p = torch.randn(256, 256, device="cuda"); (_p @ _p).sum().item()
        torch.cuda.synchronize(); GPU_OK = True
    except RuntimeError as e:
        print(f"GPU unusable: {str(e).splitlines()[0]}")
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

splits = {s: [d for d in dataset if d.split == s] for s in ("train", "val", "test")}
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} "
      f"(seq {N_SEQ} + topo {TOPO_DIM}) | pos_weight {blob['pos_weight']:.2f}")
for s, ds in splits.items():
    n = sum(int(d.num_nodes) for d in ds); p = sum(int(d.y.sum()) for d in ds)
    print(f"  {s:<6} {len(ds):>3} chains {n:>7,} residues {p:>6,} pos ({100*p/max(n,1):.2f}%)")

## Cell 2 — Equivariant layers

`EGCL` is a faithful E(n)-equivariant convolution. Two details that are easy to
get wrong and that break equivariance silently:

- **Only the squared distance enters the MLPs.** Feeding a coordinate component
  anywhere into `edge_mlp` destroys rotation invariance.
- **Coordinate updates are normalised by degree**, not summed raw. With a mean
  degree near 10 and a wide spread, unnormalised sums make coordinate updates
  scale with local density and training diverges.

In [ ]:
class EGCL(nn.Module):
    """One E(n)-equivariant graph convolution (Satorras et al., 2021)."""

    def __init__(self, h_dim, edge_attr_dim=2, act=nn.SiLU, update_coords=True):
        super().__init__()
        self.update_coords = update_coords
        # message input: h_i, h_j, ||x_i-x_j||^2, edge_attr  -- all INVARIANT
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * h_dim + 1 + edge_attr_dim, h_dim), act(),
            nn.Linear(h_dim, h_dim), act())
        self.node_mlp = nn.Sequential(
            nn.Linear(2 * h_dim, h_dim), act(), nn.Linear(h_dim, h_dim))
        self.att = nn.Sequential(nn.Linear(h_dim, 1), nn.Sigmoid())
        if update_coords:
            self.coord_mlp = nn.Sequential(
                nn.Linear(h_dim, h_dim), act(), nn.Linear(h_dim, 1, bias=False))
            nn.init.xavier_uniform_(self.coord_mlp[-1].weight, gain=0.001)  # near-identity start
        self.norm = nn.LayerNorm(h_dim)

    def forward(self, h, x, edge_index, edge_attr):
        src, dst = edge_index
        rel = x[src] - x[dst]                       # equivariant direction
        d2 = (rel ** 2).sum(-1, keepdim=True)       # invariant scalar

        m = self.edge_mlp(torch.cat([h[src], h[dst], d2, edge_attr], dim=-1))
        m = m * self.att(m)                         # soft edge gating

        agg = torch.zeros_like(h).index_add_(0, dst, m)
        deg = torch.zeros(h.size(0), 1, device=h.device).index_add_(
            0, dst, torch.ones_like(d2)).clamp(min=1)

        h = h + self.node_mlp(torch.cat([h, agg / deg], dim=-1))
        h = self.norm(h)

        if self.update_coords:
            # normalise direction: raw ||rel|| makes the update scale with distance
            w = self.coord_mlp(m) * rel / (d2.sqrt() + 1.0)
            x = x + torch.zeros_like(x).index_add_(0, dst, w) / deg
        return h, x


class TopoFusion(nn.Module):
    """Gated cross-attention from node state to its topological descriptor."""

    def __init__(self, h_dim, topo_dim):
        super().__init__()
        self.proj = nn.Linear(topo_dim, h_dim)
        self.gate = nn.Sequential(nn.Linear(h_dim + topo_dim, h_dim), nn.Sigmoid())

    def forward(self, h, t):
        g = self.gate(torch.cat([h, t], dim=-1))
        self.last_gate = g.detach()                 # inspectable after training
        return h + g * self.proj(t)

## Cell 3 — The model

In [ ]:
class TopologicalEGNN(nn.Module):
    def __init__(self, n_seq, topo_dim, h_dim=128, n_layers=4,
                 edge_attr_dim=2, use_topo=True, dropout=0.1):
        super().__init__()
        self.n_seq, self.topo_dim, self.use_topo = n_seq, topo_dim, use_topo
        in_dim = n_seq + (topo_dim if use_topo else 0)
        self.embed = nn.Sequential(nn.Linear(in_dim, h_dim), nn.SiLU(),
                                   nn.Linear(h_dim, h_dim))
        self.layers = nn.ModuleList([
            EGCL(h_dim, edge_attr_dim, update_coords=(i < n_layers - 1))
            for i in range(n_layers)])
        self.fusions = nn.ModuleList([
            TopoFusion(h_dim, topo_dim) if use_topo else nn.Identity()
            for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.Linear(h_dim, h_dim // 2), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(h_dim // 2, 1))

    def forward(self, data):
        x_all = data.x
        seq, topo = x_all[:, :self.n_seq], x_all[:, self.n_seq:]
        h = self.embed(x_all if self.use_topo else seq)
        pos = data.pos
        for layer, fuse in zip(self.layers, self.fusions):
            h, pos = layer(h, pos, data.edge_index, data.edge_attr)
            if self.use_topo:
                h = fuse(h, topo)
        return self.head(h).squeeze(-1)


m = TopologicalEGNN(N_SEQ, TOPO_DIM).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in m.parameters()):,}")

## Cell 4 — Equivariance test **before** training

This is the single most important cell in the notebook. It verifies the model's
central claim on random weights, where a failure is unambiguous:

- **Invariance of predictions.** Rotate + translate the input; logits must be
  identical. Any deviation beyond float tolerance means a coordinate leaked into
  a scalar path.
- **Equivariance of coordinates.** The internal coordinate updates must satisfy
  $\text{model}(Rx + t) = R\,\text{model}(x) + t$.

Testing before training matters: a trained model can *learn* approximate
invariance from data and mask a structural bug.

In [ ]:
@torch.no_grad()
def equivariance_report(model, data, n_trials=10):
    model.eval()
    d = data.clone().to(DEVICE)
    base = model(d).cpu()

    worst_inv, worst_equi = 0.0, 0.0
    for s in range(n_trials):
        g = torch.Generator().manual_seed(s)
        Q, _ = torch.linalg.qr(torch.randn(3, 3, generator=g))
        Q = (Q * torch.sign(torch.det(Q))).to(DEVICE)
        t = (torch.randn(3, generator=g) * 25.0).to(DEVICE)

        d2 = d.clone()
        d2.pos = d.pos @ Q.T + t
        worst_inv = max(worst_inv, float((model(d2).cpu() - base).abs().max()))

        # coordinate-channel equivariance, probed through one layer
        h = model.embed(d.x)
        h1, x1 = model.layers[0](h, d.pos, d.edge_index, d.edge_attr)
        _, x2 = model.layers[0](h, d.pos @ Q.T + t, d.edge_index, d.edge_attr)
        worst_equi = max(worst_equi, float((x2 - (x1 @ Q.T + t)).abs().max()))

    return worst_inv, worst_equi


probe = splits["train"][0]
inv, equi = equivariance_report(TopologicalEGNN(N_SEQ, TOPO_DIM).to(DEVICE), probe)
print(f"prediction invariance : max |dlogit| = {inv:.3e}")
print(f"coordinate equivariance: max error   = {equi:.3e}")
assert inv < 1e-3, "MODEL IS NOT ROTATION INVARIANT"
assert equi < 1e-3, "COORDINATE UPDATES ARE NOT EQUIVARIANT"
print("E(3) guarantees hold")

## Cell 5 — Training loop

Class imbalance is handled with `pos_weight` in `BCEWithLogitsLoss` (~9.9), which
is numerically stabler than weighting a post-sigmoid BCE. Model selection is on
**validation PR-AUC**, not loss or accuracy: at a 9 % positive rate, accuracy is
meaningless and PR-AUC is the metric that reflects the retrieval task.

In [ ]:
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

POS_WEIGHT = float(blob["pos_weight"])


def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    tot, n, ps, ys = 0.0, 0, [], []
    crit = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    for batch in loader:
        batch = batch.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits = model(batch)
            loss = crit(logits, batch.y)
        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        tot += float(loss) * batch.num_nodes; n += batch.num_nodes
        ps.append(torch.sigmoid(logits).detach().cpu().numpy())
        ys.append(batch.y.detach().cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    return {"loss": tot / n, "roc": roc_auc_score(y, p),
            "pr": average_precision_score(y, p), "p": p, "y": y}


def train_model(use_topo, epochs=120, h_dim=128, n_layers=4, lr=1e-3, tag=""):
    torch.manual_seed(SEED)
    model = TopologicalEGNN(N_SEQ, TOPO_DIM, h_dim, n_layers, use_topo=use_topo).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    tl = DataLoader(splits["train"], batch_size=8, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=8)

    best, best_state, patience = -1.0, None, 0
    for ep in range(epochs):
        tr = run_epoch(model, tl, opt)
        va = run_epoch(model, vl)
        sched.step()
        if va["pr"] > best:
            best, best_state, patience = va["pr"], copy.deepcopy(model.state_dict()), 0
        else:
            patience += 1
        if ep % 10 == 0 or ep == epochs - 1:
            print(f"  [{tag}] ep {ep:3d} | train loss {tr['loss']:.4f} pr {tr['pr']:.3f} "
                  f"| val roc {va['roc']:.3f} pr {va['pr']:.3f}")
        if patience >= 30:
            print(f"  [{tag}] early stop at epoch {ep}")
            break
    model.load_state_dict(best_state)
    return model, best

## Cell 6 — Three-arm ablation

The comparison that decides the project's claim, now on genuine
sequence-clustered splits:

| arm | inputs |
|---|---|
| `seq-only` | one-hot + dihedrals, EGNN, no topology |
| `full` | + all 33 topological features via gated fusion |
| `burial-only` | + only the two burial features, topology zeroed |

`burial-only` is the arm that matters. It isolates whether the gain comes from
*persistent homology* or merely from the density proxies riding along with it.

In [ ]:
def threshold_metrics(p, y):
    """MCC at the F1-optimal threshold, plus that threshold."""
    best_mcc, best_t, best_f1 = -1.0, 0.5, 0.0
    for t in np.linspace(0.05, 0.95, 91):
        pred = (p >= t).astype(int)
        if pred.sum() == 0:
            continue
        tp = int(((pred == 1) & (y == 1)).sum())
        prec = tp / max(pred.sum(), 1); rec = tp / max(y.sum(), 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-9)
        if f1 > best_f1:
            best_f1, best_t, best_mcc = f1, t, matthews_corrcoef(y, pred)
    return best_mcc, best_f1, best_t


ARMS = {"seq-only": "none", "burial-only": "burial", "full": "all"}
results, models = {}, {}

for arm, mode in ARMS.items():
    print(f"\n=== arm: {arm} ===")
    ds_backup = [d.x.clone() for d in dataset]
    if mode != "all":
        b_idx = [TOPO_NAMES.index(n) for n in blob["burial_names"]]
        for d in dataset:
            t = d.x[:, N_SEQ:].clone()
            if mode == "none":
                t.zero_()
            else:                                   # burial only: zero the rest
                keep = torch.zeros_like(t); keep[:, b_idx] = t[:, b_idx]; t = keep
            d.x = torch.cat([d.x[:, :N_SEQ], t], dim=1)

    model, best_val = train_model(use_topo=True, tag=arm)
    te = run_epoch(model, DataLoader(splits["test"], batch_size=8))
    mcc, f1, thr = threshold_metrics(te["p"], te["y"])
    results[arm] = {"val_pr": best_val, "test_roc": te["roc"], "test_pr": te["pr"],
                    "test_mcc": mcc, "test_f1": f1, "thr": thr}
    models[arm] = model
    print(f"  -> test ROC-AUC {te['roc']:.3f} | PR-AUC {te['pr']:.3f} "
          f"| MCC {mcc:.3f} | F1 {f1:.3f} @ thr {thr:.2f}")

    for d, xb in zip(dataset, ds_backup):
        d.x = xb

print("\n" + "=" * 74)
print(f"{'arm':<14}{'val PR':>9}{'test ROC':>10}{'test PR':>10}{'test MCC':>10}{'test F1':>9}")
for a, r in results.items():
    print(f"{a:<14}{r['val_pr']:>9.3f}{r['test_roc']:>10.3f}{r['test_pr']:>10.3f}"
          f"{r['test_mcc']:>10.3f}{r['test_f1']:>9.3f}")
print(f"\ntopology beyond burial: PR-AUC "
      f"{results['full']['test_pr'] - results['burial-only']['test_pr']:+.4f}")
print(f"topology beyond sequence: PR-AUC "
      f"{results['full']['test_pr'] - results['seq-only']['test_pr']:+.4f}")

## Cell 7 — Save weights and inspect the learned gates

In [ ]:
out = Path("/kaggle/working"); (out / "models").mkdir(parents=True, exist_ok=True)
for arm, model in models.items():
    torch.save({"state_dict": model.state_dict(), "arm": arm,
                "n_seq": N_SEQ, "topo_dim": TOPO_DIM,
                "metrics": results[arm], "topo_names": TOPO_NAMES},
               out / "models" / f"topological_egnn_{arm}.pt")
    print(f"saved models/topological_egnn_{arm}.pt")
(out / "phase5_metrics.json").write_text(json.dumps(results, indent=2, default=float))

# --- what did the fusion gate actually learn? --------------------------------
model = models["full"]; model.eval()
with torch.no_grad():
    _ = model(splits["test"][0].clone().to(DEVICE))
gates = [f.last_gate.mean().item() for f in model.fusions if hasattr(f, "last_gate")]
print(f"\nmean fusion gate per layer: {[round(g, 3) for g in gates]}")
print("  (near 0 = topology ignored; higher = actively used)")

---

## Reading the ablation

**`full` − `burial-only`** is the headline. It is the strongest statement the
project can make, because it holds sequence, architecture, split, and burial
fixed and varies only the persistent homology.

**Expect noise.** The test split is a handful of clusters. A PR-AUC difference
smaller than the seed-to-seed spread is not a result — if the ablation gap is
small, re-run all three arms across 3–5 seeds and report mean ± std before
claiming anything. That is the difference between an ISEF project that survives
questioning and one that does not.

**The gate values are evidence too.** If mean gates sit near zero, the network
learned to ignore topology regardless of what the ablation deltas suggest.